In [1]:
!nvidia-smi

Tue Jun 30 12:39:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -U ultralytics -q
!pip install pyyaml roboflow -q

import ultralytics
ultralytics.checks()
# Expected: Ultralytics 8.3.0 ✅  Python-3.x  torch-2.x  CUDA:0

Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 6960.6/8062.4 GB disk)


In [5]:
import os
from pathlib import Path

# ── EDIT THIS ─────────────────────────────────────────────────────────────
DATASET_SLUG = 'datasets'   # ← your Kaggle dataset slug
# ──────────────────────────────────────────────────────────────────────────

KAGGLE_INPUT = f'/kaggle/input/{DATASET_SLUG}'

# Auto-detect inner folder name (coco, unified_4class_dataset, etc.)
DATASET_ROOT = None
for item in os.listdir(KAGGLE_INPUT):
    candidate = os.path.join(KAGGLE_INPUT, item)
    if os.path.isdir(candidate):
        DATASET_ROOT = candidate
        break

if DATASET_ROOT is None:
    DATASET_ROOT = KAGGLE_INPUT   # flat structure, no subfolder

print(f"Dataset root: {DATASET_ROOT}")
print("Contents:")
for item in os.listdir(DATASET_ROOT):
    print(f"  {item}")

Dataset root: /kaggle/input/datasets/starlinpereira
Contents:
  unified-4class-dataset


In [4]:
import os
if os.path.exists('/kaggle/input'):
    print("📁 Folders found in /kaggle/input:")
    print(os.listdir('/kaggle/input'))
else:
    print("❌ /kaggle/input directory does not exist yet. Did you turn on the session accelerator?")

📁 Folders found in /kaggle/input:
['datasets']


In [ ]:
from pathlib import Path
from collections import Counter

CLASS_NAMES  = ['person', 'bag', 'knife', 'pistol']
DATASET_ROOT = '/kaggle/input/datasets/starlinpereira/unified-4class-dataset/coco'

folders = {
    'train images' : DATASET_ROOT + '/train/images',
    'train labels' : DATASET_ROOT + '/train/labels',
    'val images'   : DATASET_ROOT + '/valid/images',
    'val labels'   : DATASET_ROOT + '/valid/labels',
}

print(f"Dataset root: {DATASET_ROOT}")
print("-" * 60)
all_ok = True
for name, path in folders.items():
    p = Path(path)
    if p.exists():
        count = len(list(p.glob("*")))
        print(f"  ✓  {name:<18}  {count} files")
    else:
        alt = path.replace('/valid/', '/val/')
        p2  = Path(alt)
        if p2.exists():
            count = len(list(p2.glob("*")))
            print(f"  ✓  {name:<18}  {count} files  (folder: val/)")
        else:
            print(f"  ✗  {name:<18}  MISSING")
            all_ok = False

lbl_dir = Path(DATASET_ROOT) / 'train' / 'labels'
counts  = Counter()
for lbl in lbl_dir.glob('*.txt'):
    with open(lbl) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                counts[int(parts[0])] += 1

print("\nClass distribution in train set:")
for cid, name in enumerate(CLASS_NAMES):
    bar = '█' * (counts[cid] // 50)
    print(f"  {cid} {name:<10} {counts[cid]:>6} instances  {bar}")

print("-" * 60)
print("✅ Ready to train." if all_ok else "❌ Fix missing folders before continuing.")


Dataset root: /kaggle/input/datasets/starlinpereira/unified-4class-dataset/coco
------------------------------------------------------------
  ✓  train images        11889 files
  ✓  train labels        11889 files
  ✓  val images          1847 files  (folder: val/)
  ✓  val labels          1847 files  (folder: val/)


In [16]:
import yaml
from pathlib import Path

DATASET_ROOT = '/kaggle/input/datasets/starlinpereira/unified-4class-dataset/coco'
val_folder   = 'val'   # confirmed from Cell 4

YAML_OUT = '/kaggle/working/data.yaml'
data = {
    'path'  : DATASET_ROOT,
    'train' : 'train/images',
    'val'   : f'{val_folder}/images',
    'nc'    : 4,
    'names' : {
        0: 'person',
        1: 'bag',
        2: 'knife',
        3: 'pistol',
    }
}

with open(YAML_OUT, 'w') as f:
    yaml.dump(data, f, default_flow_style=False, allow_unicode=True)

print(f"data.yaml written → {YAML_OUT}")
with open(YAML_OUT) as f:
    print(f.read())


data.yaml written → /kaggle/working/data.yaml
names:
  0: person
  1: bag
  2: knife
  3: pistol
nc: 4
path: /kaggle/input/datasets/starlinpereira/unified-4class-dataset/coco
train: train/images
val: val/images



In [17]:
from ultralytics import YOLO
from pathlib import Path
import shutil, os

YAML_PATH  = '/kaggle/working/data.yaml'
OUTPUT_DIR = '/kaggle/working/unified_4class_training'
os.makedirs(OUTPUT_DIR, exist_ok=True)

_saved = set()
def auto_checkpoint(trainer):
    epoch = trainer.epoch + 1
    if epoch % 10 == 0 and epoch not in _saved:
        src  = Path(trainer.save_dir) / 'weights' / 'last.pt'
        dest = f'/kaggle/working/checkpoint_epoch{epoch:03d}.pt'
        if src.exists():
            shutil.copy(src, dest)
            _saved.add(epoch)
            print(f"\n  💾  Checkpoint saved → checkpoint_epoch{epoch:03d}.pt\n")

model = YOLO('yolo11s.pt')
model.add_callback('on_train_epoch_end', auto_checkpoint)

print("Starting YOLO11s 4-class training...")
print("Classes : person (0) · bag (1) · knife (2) · pistol (3)")
print("=" * 60)

results = model.train(
    data         = YAML_PATH,
    epochs       = 150,
    patience     = 30,
    imgsz        = 416,
    batch        = 32,
    device       = 0,
    workers      = 4,
    optimizer    = 'AdamW',
    lr0          = 0.001,
    lrf          = 0.01,
    momentum     = 0.937,
    weight_decay = 0.0005,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=20.0, translate=0.1, scale=0.6,
    shear=3.0, perspective=0.0001,
    fliplr=0.5, flipud=0.0,
    mosaic=1.0, mixup=0.15,
    copy_paste=0.10, erasing=0.4,
    project      = OUTPUT_DIR,
    name         = 'unified_yolo11s',
    exist_ok     = True,
    save_period  = 10,
    plots        = True,
    verbose      = True,
    pretrained   = True,
)

print("\n✅ Training complete!")
print(f"Best weights: {OUTPUT_DIR}/unified_yolo11s/weights/best.pt")


Starting YOLO11s 4-class training...
Classes : person (0) · bag (1) · knife (2) · pistol (3)
Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=20.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=unified_

In [18]:
from ultralytics import YOLO
from pathlib import Path

CLASS_NAMES = ['person', 'bag', 'knife', 'pistol']
YAML_PATH   = '/kaggle/working/data.yaml'
OUTPUT_DIR  = '/kaggle/working/unified_4class_training'

for candidate in [
    f'{OUTPUT_DIR}/unified_yolo11s/weights/best.pt',
    f'{OUTPUT_DIR}/unified_yolo11s_resumed/weights/best.pt',
]:
    if Path(candidate).exists():
        BEST_PT = candidate
        break

print(f"Evaluating: {BEST_PT}")
model   = YOLO(BEST_PT)
metrics = model.val(data=YAML_PATH, imgsz=416, device=0,
                    conf=0.50, iou=0.50, plots=True, verbose=False)

print("\n" + "=" * 55)
print("  YOLO11s 4-CLASS — FINAL METRICS")
print("=" * 55)
print(f"  mAP@0.5        : {metrics.box.map50:.4f}  (target > 0.85)")
print(f"  mAP@0.5:0.95   : {metrics.box.map:.4f}  (target > 0.60)")
print(f"  Precision      : {metrics.box.mp:.4f}  (target > 0.85)")
print(f"  Recall         : {metrics.box.mr:.4f}  (target > 0.80)")
print("=" * 55)
print("\n  Per-class AP@0.5:")
print(f"  {'Class':<12} {'AP':>6}  Progress")
print("  " + "-" * 40)
for name, ap in zip(CLASS_NAMES, metrics.box.ap50):
    bar  = '█' * int(ap * 25)
    flag = '  ← may need more data' if ap < 0.70 else ''
    print(f"  {name:<12} {ap:.4f}  {bar}{flag}")

Evaluating: /kaggle/working/unified_4class_training/unified_yolo11s/weights/best.pt
Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 101 layers, 9,414,348 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 68.9±82.6 MB/s, size: 98.0 KB)
val: Scanning /kaggle/input/datasets/starlinpereira/unified-4class-dataset/coco/val/labels... 1847 images, 7 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1847/1847 276.8it/s 6.7s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/starlinpereira/unified-4class-dataset/coco/val is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 116/116 8.9it/s 13.0s<0.1s
                   all       1847       6400      0.858      0.603      0.585      0.429
Speed: 0.4ms preprocess, 3.8ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to /kaggle/working/runs/detect/v